# Drug-Drug Interaction — Model Training Walkthrough
This notebook walks through the training pipeline for all three models.

In [ ]:
import torch
import sys

sys.path.append("..")
from src.model import MLPBaseline, GCNModel, GATModel
from src.dataset import load_twosides, build_drug_vocab
from torch_geometric.utils import negative_sampling
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F

print("Imports done")

## 1. Load Graph

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

data = torch.load("../data/processed/ddi_graph.pt", weights_only=False)
x = data.x.to(device)
edge_index = data.edge_index.to(device)

print(f"Nodes:            {data.num_nodes}")
print(f"Node feature dim: {data.x.shape[1]}")
print(f"Edges:            {data.num_edges}")

## 2. Edge Splitting
We split edges into train/val/test at the **edge level** — not node level. This simulates predicting interactions between drugs that are already in the graph but whose interaction is unknown.

In [ ]:
def split_edges(edge_index, val_ratio=0.1, test_ratio=0.1):
    mask = edge_index[0] < edge_index[1]
    unique_edges = edge_index[:, mask]
    num_edges = unique_edges.shape[1]
    indices = torch.randperm(num_edges)

    num_val = int(num_edges * val_ratio)
    num_test = int(num_edges * test_ratio)

    val_idx = indices[:num_val]
    test_idx = indices[num_val : num_val + num_test]
    train_idx = indices[num_val + num_test :]

    return (
        unique_edges[:, train_idx],
        unique_edges[:, val_idx],
        unique_edges[:, test_idx],
    )


train_edges, val_edges, test_edges = split_edges(edge_index)
print(f"Train edges: {train_edges.shape[1]}")
print(f"Val edges:   {val_edges.shape[1]}")
print(f"Test edges:  {test_edges.shape[1]}")

## 3. Training Loop
We train all 3 models and track validation AUC per epoch to visualize learning curves.

In [ ]:
def evaluate(model, x, edge_index, pos_edges, device, num_nodes):
    model.eval()
    with torch.no_grad():
        neg_edges = negative_sampling(edge_index, num_nodes, pos_edges.shape[1]).to(
            device
        )
        pos_scores = (
            torch.sigmoid(model(x, edge_index, pos_edges.to(device))).cpu().numpy()
        )
        neg_scores = torch.sigmoid(model(x, edge_index, neg_edges)).cpu().numpy()
        scores = np.concatenate([pos_scores, neg_scores])
        labels = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
        return roc_auc_score(labels, scores)


def train_model(model, model_name, epochs=50):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    num_nodes = x.shape[0]
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        neg_edges = negative_sampling(edge_index, num_nodes, train_edges.shape[1]).to(
            device
        )
        pos_scores = model(x, edge_index, train_edges.to(device))
        neg_scores = model(x, edge_index, neg_edges)
        scores = torch.cat([pos_scores, neg_scores])
        labels = torch.cat(
            [torch.ones(pos_scores.shape[0]), torch.zeros(neg_scores.shape[0])]
        ).to(device)
        loss = F.binary_cross_entropy_with_logits(scores, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 5 == 0:
            val_auc = evaluate(model, x, edge_index, val_edges, device, num_nodes)
            history.append((epoch, loss.item(), val_auc))
            print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | Val AUC: {val_auc:.4f}")

    return history


print("Training MLP...")
mlp = MLPBaseline(input_dim=2048).to(device)
mlp_history = train_model(mlp, "mlp", epochs=50)

In [ ]:
print("Training GCN...")
gcn = GCNModel(input_dim=2048).to(device)
gcn_history = train_model(gcn, "gcn", epochs=50)

In [ ]:
print("Training GAT...")
gat = GATModel(input_dim=2048).to(device)
gat_history = train_model(gat, "gat", epochs=50)

## 4. Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for history, label, color in [
    (mlp_history, "MLP", "#2ecc71"),
    (gcn_history, "GCN", "#3498db"),
    (gat_history, "GAT", "#9b59b6"),
]:
    epochs = [h[0] for h in history]
    losses = [h[1] for h in history]
    aucs = [h[2] for h in history]
    axes[0].plot(epochs, losses, label=label, color=color)
    axes[1].plot(epochs, aucs, label=label, color=color)

axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE Loss")
axes[0].legend()

axes[1].set_title("Validation ROC-AUC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("ROC-AUC")
axes[1].legend()

plt.suptitle("Training Curves — DDI Prediction", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../data/processed/training_curves.png", dpi=150)
plt.show()

## 5. Final Test Evaluation

In [ ]:
import pandas as pd

num_nodes = x.shape[0]

rows = []
for model, name in [(mlp, "MLP"), (gcn, "GCN"), (gat, "GAT")]:
    model.eval()
    with torch.no_grad():
        neg_test = negative_sampling(edge_index, num_nodes, test_edges.shape[1]).to(
            device
        )
        pos_scores = (
            torch.sigmoid(model(x, edge_index, test_edges.to(device))).cpu().numpy()
        )
        neg_scores = torch.sigmoid(model(x, edge_index, neg_test)).cpu().numpy()
        scores = np.concatenate([pos_scores, neg_scores])
        labels = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
        auc = roc_auc_score(labels, scores)
        ap = average_precision_score(labels, scores)
        rows.append(
            {"Model": name, "ROC-AUC": round(auc, 4), "Avg Precision": round(ap, 4)}
        )

results = pd.DataFrame(rows)
print(results.to_string(index=False))
print("\nKey finding: MLP outperforms GNNs — over-smoothing on dense graph")